# A brain that teaches itself to read and to talk

Built from commit `8593363+local-edits`. No pretrained weights, no backpropagation. Every reply is a choice made by a
striatum whose synapses change by a three-factor (dopamine-gated delta) rule; the brain reads Wikipedia by
curiosity, practises on a teacher's questions and on questions it asks itself, sleeps, and sits held-out exams
(SQuAD v1.1 dev, DailyDialog test) every simulated day. See `PREREGISTRATION_selftrain.md`.

In [ ]:
import os, sys, time, platform, multiprocessing
W = "/kaggle/working/brain"
START = time.time()
print(sys.version, "| cores:", multiprocessing.cpu_count())
for d in ("brain", "chat", "tests"):
    os.makedirs(os.path.join(W, d), exist_ok=True)
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
%%writefile /kaggle/working/brain/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/brain/brain/reading.py
"""Reading to answer: which stretch of what was read answers the question.

The old reply path found a page and said its first sentence with a phrase in front of it. Nothing learned was
deciding anything. Here the answer is a *choice* - every short stretch of the most relevant sentences is a
candidate, cortex describes each one (what it looks like, what surrounds it, how close it sits to the words the
question used), and the striatum picks. What it learns from being told the right answer is which descriptions
go with right answers for which kind of question: that "when" wants a stretch shaped like a date, that the
answer to "who" usually sits right next to the verb the question used, that an answer rarely repeats the
question's own words.

This is the logistic-regression reader of the SQuAD paper (Rajpurkar et al. 2016, 51 F1) without its parser:
no part-of-speech tags and no dependency paths, because nothing here was given a grammar. Shape and position
stand in for them, and the gap between this and 51 is the price of that.

Relevance of a sentence is weighted by how rare each matched word is in everything read so far - a count this
brain keeps, not a constant - and words match on a crude stem so "appeared" finds "appear".
"""
import collections
import math
import re
import string

TOKEN = re.compile(r"\w+(?:[-'’.]\w+)*|[^\w\s]")
STOP = set("""a an the is are was were be been being am of to in on at by for with from into onto over under about
through and or but if then than as so because while when where that which who whom whose what why how it its they
them their he she his her him we us our you your i me my this these those there here also not no do does did can
could will would may might must should shall has have had having one ones such other same very many much more most
some any all each both either neither own only just still now later after before during between among against
within without s 's ’s""".split())
WH = ("what", "which", "who", "whom", "whose", "when", "where", "why", "how")
PUNCT_EDGE = set(",.;:!?)(]['\"`’‘“”-")
SUFFIXES = ("ational", "ization", "ations", "ingly", "ments", "ness", "ment", "ings", "ing", "ies", "ied",
            "edly", "ed", "es", "ly", "s")


def stem(word):
    w = word.lower()
    for suf in SUFFIXES:
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            w = w[: -len(suf)]
            break
    return w[:8]


def tokens(text):
    """-> [(word, start_char, end_char)]"""
    return [(m.group(), m.start(), m.end()) for m in TOKEN.finditer(text)]


def shape(word):
    if word[0].isdigit():
        return "d"
    if word[0].isupper():
        return "X"
    if word[0].isalpha():
        return "x"
    return "p"


def squash(shapes):
    out = []
    for s in shapes:
        if not out or out[-1] != s:
            out.append(s)
    return "".join(out)[:5]


def bucket(x, edges=(0, 1, 2, 3, 5, 8)):
    for i, e in enumerate(edges):
        if x <= e:
            return str(i)
    return str(len(edges))


# ------------------------------------------------------------------ official SQuAD scoring
def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = collections.Counter(p) & collections.Counter(g)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision, recall = same / len(p), same / len(g)
    return 2 * precision * recall / (precision + recall)


def exact(pred, gold):
    return float(normalize(pred) == normalize(gold))


def best(pred, golds, metric):
    return max(metric(pred, g) for g in golds) if golds else 0.0


# ------------------------------------------------------------------ word rarity, kept by reading
class Rarity:
    """How rare each stem is in what has been read. Grows with reading; nothing is set by hand."""

    def __init__(self):
        self.docs = 0
        self.df = collections.Counter()

    def read(self, text):
        self.docs += 1
        self.df.update({stem(w) for w, _, _ in tokens(text) if w[0].isalnum()})

    def idf(self, s):
        return math.log((self.docs + 1) / (self.df.get(s, 0) + 1)) + 1.0


# ------------------------------------------------------------------ the question
class Question:
    def __init__(self, text, rarity):
        self.text = text
        toks = [w for w, _, _ in tokens(text)]
        low = [w.lower() for w in toks]
        self.wh, self.kind, self.focus = "none", "none", "none"
        for i, w in enumerate(low):
            if w in WH:
                self.wh = w
                nxt = low[i + 1] if i + 1 < len(low) else ""
                if w == "how" and nxt:
                    self.wh = "how " + nxt if nxt in ("many", "much", "long", "old", "far", "large", "big",
                                                     "often", "tall", "high") else "how"
                self.kind = f"{self.wh} {nxt}" if nxt else self.wh
                # the word that says what sort of thing is wanted: "what *year*", "which *river*"
                for v in low[i + 1:i + 4]:
                    if v not in STOP and v.isalpha():
                        self.focus = stem(v)
                        break
                break
        self.stems = {stem(w) for w in low if w not in STOP and w[0].isalnum()}
        self.weight = {s: rarity.idf(s) for s in self.stems}
        self.total = sum(self.weight.values()) or 1.0
        self.bigrams = {(stem(a), stem(b)) for a, b in zip(low, low[1:]) if a[0].isalnum() and b[0].isalnum()}
        # the question's verb, roughly: the first lowercase content word after the wh-word
        self.verb = "none"
        seen_wh = False
        for w, lw in zip(toks, low):
            if lw in WH:
                seen_wh = True
                continue
            if seen_wh and lw not in STOP and w[0].islower() and w.isalpha():
                self.verb = stem(w)
                break


# ------------------------------------------------------------------ the passage
class Passage:
    def __init__(self, text):
        self.text = text
        self.toks = tokens(text)
        self.words = [w for w, _, _ in self.toks]
        self.low = [w.lower() for w in self.words]
        self.stems = [stem(w) for w in self.words]
        self.sentences = []
        start = 0
        for i, w in enumerate(self.words):
            end_here = w in (".", "?", "!") and (i + 1 == len(self.words) or self.words[i + 1][0].isupper()
                                                  or self.words[i + 1] in ('"', "“"))
            if end_here or i + 1 == len(self.words):
                self.sentences.append((start, i + 1))
                start = i + 1


class Reader:
    """Proposes the candidates and describes them. The striatum does the choosing."""

    def __init__(self, rarity=None, max_len=10, top_sentences=3):
        self.rarity = rarity or Rarity()
        self.max_len = max_len
        self.top_sentences = top_sentences

    def rank_sentences(self, q, p):
        scored = []
        for si, (a, b) in enumerate(p.sentences):
            st = set(p.stems[a:b])
            score = sum(q.weight.get(s, 0.0) for s in st)
            bi = {(x, y) for x, y in zip(p.stems[a:b], p.stems[a + 1:b])}
            score += 0.5 * sum(1 for g in q.bigrams if g in bi)
            scored.append((score, si))
        scored.sort(reverse=True)
        return scored

    def candidates(self, q, p):
        """-> list of (start_tok, end_tok_exclusive, features)."""
        ranked = self.rank_sentences(q, p)
        top = ranked[0][0] if ranked else 0.0
        W, K, F = q.wh, q.kind, q.focus
        out = []
        for rank, (score, si) in enumerate(ranked[: self.top_sentences]):
            a, b = p.sentences[si]
            matched = [i for i in range(a, b) if p.stems[i] in q.stems]
            frac = score / top if top > 0 else 0.0
            sent_feats = [f"srank|{rank}", f"sfrac|{bucket(int(frac * 10), (2, 4, 6, 8, 9))}",
                          f"srank_wh|{W}|{rank}"]
            for i in range(a, b):
                if p.words[i][0] in PUNCT_EDGE and p.words[i] not in ("$",):
                    continue
                for j in range(i + 1, min(b, i + self.max_len) + 1):
                    last = p.words[j - 1]
                    if last[0] in PUNCT_EDGE and last not in (")", "%"):
                        continue
                    span_low = p.low[i:j]
                    content = [k for k in range(i, j) if p.low[k] not in STOP and p.words[k][0].isalnum()]
                    if not content:
                        continue
                    inq = sum(1 for k in content if p.stems[k] in q.stems) / len(content)
                    if inq == 1.0:
                        continue
                    out.append((i, j, sent_feats + self._describe(q, p, i, j, a, b, matched, inq, W, K, F)))
        return out

    def _describe(self, q, p, i, j, a, b, matched, inq, W, K, F):
        L = j - i
        shapes = [shape(w) for w in p.words[i:j]]
        sq = squash(shapes)
        first, last = p.low[i], p.low[j - 1]
        left = p.low[i - 1] if i > a else "<s>"
        right = p.low[j] if j < b else "</s>"
        left2 = p.low[i - 2] if i - 1 > a else "<s>"
        lq = stem(left) in q.stems
        rq = stem(right) in q.stems
        # matched question words either side of the span, near and far
        near_l = {p.stems[k] for k in matched if i - 5 <= k < i}
        near_r = {p.stems[k] for k in matched if j <= k < j + 5}
        wl = sum(q.weight[s] for s in near_l) / q.total
        wr = sum(q.weight[s] for s in near_r) / q.total
        wall = sum(q.weight[s] for s in (near_l | near_r)) / q.total
        before = [k for k in matched if k < i]
        after = [k for k in matched if k >= j]
        dist_l = i - before[-1] if before else 99
        dist_r = after[0] - j + 1 if after else 99
        where = "none" if not matched else ("between" if before and after else ("after" if before else "before"))
        verb_near = any(p.stems[k] == q.verb for k in range(max(a, i - 6), min(b, j + 6)))
        inner_comma = "," in p.words[i:j]
        paren = left == "(" or right == ")"
        feats = [
            f"b|{W}", f"len|{W}|{min(L, 8)}", f"len|{min(L, 8)}",
            f"shape|{W}|{sq}", f"shape|{K}|{sq}", f"shapeF|{F}|{sq}", f"shape|{sq}",
            f"sh1|{W}|{shapes[0]}|{shapes[-1]}",
            f"first|{W}|{first}", f"last|{W}|{last}", f"lastF|{F}|{last}", f"firstK|{K}|{first}",
            f"left|{W}|{left}", f"right|{W}|{right}", f"left2|{W}|{left2}|{left}",
            f"left|{left}", f"right|{right}",
            f"lq|{W}|{lq}|{rq}", f"lq|{lq}|{rq}",
            f"dl|{W}|{bucket(dist_l)}", f"dr|{W}|{bucket(dist_r)}", f"dlr|{bucket(dist_l)}|{bucket(dist_r)}",
            f"where|{W}|{where}",
            f"inq|{W}|{int(inq * 4)}", f"inq|{int(inq * 4)}",
            f"verb|{W}|{verb_near}", f"comma|{inner_comma}|{paren}",
            f"wl|{W}|{bucket(int(wl * 10), (0, 2, 4, 6, 8))}", f"wr|{W}|{bucket(int(wr * 10), (0, 2, 4, 6, 8))}",
            (f"wall|{W}", wall), ("wall", wall), (f"wl|{W}", wl), (f"wr|{W}", wr),
        ]
        return feats

    # ---------------------------------------------------------------- one question
    def answer(self, striatum, question, passage, consolidated=True):
        q = question if isinstance(question, Question) else Question(question, self.rarity)
        p = passage if isinstance(passage, Passage) else Passage(passage)
        cands = self.candidates(q, p)
        if not cands:
            return "", None, []
        enc = striatum.encode([c[2] for c in cands])
        k, s = striatum.choose(enc, consolidated)
        i, j, _ = cands[k]
        return p.text[p.toks[i][1]:p.toks[j - 1][2]], (i, j), cands

    def practise(self, striatum, question, passage, golds):
        """One trial with feedback. Returns (F1 of what it would have said, whether the answer was reachable)."""
        q = Question(question, self.rarity)
        p = Passage(passage)
        cands = self.candidates(q, p)
        if not cands:
            return 0.0, False
        texts = [p.text[p.toks[i][1]:p.toks[j - 1][2]] for i, j, _ in cands]
        ems = [best(t, golds, exact) for t in texts]
        if max(ems) > 0:
            target = [e for e in ems]
        else:
            f1s = [best(t, golds, f1) for t in texts]
            m = max(f1s)
            if m < 0.5:
                return 0.0, False
            target = [1.0 if x == m else 0.0 for x in f1s]
        tot = sum(target)
        target = [t / tot for t in target]
        enc = striatum.encode([c[2] for c in cands])
        s = striatum.scores(enc, consolidated=False)
        said = texts[int(s.argmax())]
        striatum.teach(enc, target)
        return best(said, golds, f1), True

In [ ]:
%%writefile /kaggle/working/brain/brain/striatum.py
"""The striatum: choosing among options, and learning from how the choice turned out.

Every decision this brain makes about what to say is a choice among candidates - which stretch of a passage
answers the question, which remembered reply fits what was just said. The cortex and hippocampus supply the
candidates and describe each one; the striatum scores them and the winner is released (in the basal ganglia,
by the output nuclei disinhibiting the one action and holding the rest down).

Learning is a three-factor rule (Fremaux & Gerstner 2016): a synapse changes by

        presynaptic activity  x  postsynaptic activity  x  a neuromodulator saying "better or worse than expected"

Here the presynaptic activity is a feature of a candidate, and the third factor is dopamine: a burst for the
candidate that turned out right, a dip in proportion to how strongly each wrong one was favoured. Written out,
the change to a weight is  lr * (target_c - p_c) * feature_c  - the delta rule (Rescorla-Wagner, Widrow-Hoff)
over a softmax. It is local: every synapse needs only its own input and the error broadcast to its cell. There
are no hidden layers and nothing is backpropagated. It is also, honestly, the gradient of a one-layer log-linear
model - the point is not that it avoids that equivalence but that a one-layer rule is what the tissue can do.

Two timescales of weight (complementary learning systems). The plastic weights change on every trial; the
consolidated weights are their running average, which is what the averaged perceptron keeps and what sleep
writes back into the synapses. Exams are sat on the consolidated weights, so one lucky or unlucky trial does
not move behaviour much - the same reason a day's learning is replayed before it is trusted.

Sleep also downscales and prunes (synaptic homeostasis, Tononi & Cirelli 2014): every weight shrinks a little
and the ones that end up near zero are removed. v17 found that pruning and tuning are what separate things in
this architecture, and growth is not.
"""
import pickle
import zlib

import numpy as np


def fid(name, bits):
    """A feature's synapse. The hash is the wiring: fixed, shared by the rare features that collide."""
    return zlib.crc32(name.encode("utf-8")) & ((1 << bits) - 1)


class Striatum:
    def __init__(self, bits=22, lr=0.1, name="striatum"):
        self.name = name
        self.bits = bits
        self.lr = lr
        n = 1 << bits
        self.w = np.zeros(n, np.float64)      # plastic
        self.u = np.zeros(n, np.float64)      # time-weighted sum of changes, for the consolidated average
        self.t = 1
        self.trials = 0
        self.expected = 0.0                   # the critic: running estimate of reward, for the prediction error
        self.history = []

    # ------------------------------------------------------------ encoding
    def encode(self, candidates):
        """[[feature, ...] or [(feature, value), ...] per candidate] -> flat arrays the tissue can use."""
        ids, vals, owner = [], [], []
        for c, feats in enumerate(candidates):
            for f in feats:
                if isinstance(f, tuple):
                    name, v = f
                else:
                    name, v = f, 1.0
                ids.append(fid(name, self.bits))
                vals.append(v)
                owner.append(c)
        return (np.asarray(ids, np.int64), np.asarray(vals, np.float64), np.asarray(owner, np.int64),
                len(candidates))

    def consolidated(self):
        # the average is recomputed only after something changed: an exam asks for it thousands of times
        if getattr(self, "_cache", None) is None or self._cache[0] != (self.t, self.trials):
            self._cache = ((self.t, self.trials), self.w - self.u / self.t)
        return self._cache[1]

    def scores(self, enc, consolidated=True):
        ids, vals, owner, n = enc
        w = self.consolidated() if consolidated else self.w
        return np.bincount(owner, weights=w[ids] * vals, minlength=n)

    def choose(self, enc, consolidated=True):
        s = self.scores(enc, consolidated)
        return int(np.argmax(s)), s

    # ------------------------------------------------------------ learning
    def _apply(self, enc, err):
        ids, vals, owner, _ = enc
        delta = self.lr * err[owner] * vals
        np.add.at(self.w, ids, delta)
        np.add.at(self.u, ids, self.t * delta)
        self.t += 1
        self.trials += 1

    def teach(self, enc, target):
        """Dopamine from a known outcome: `target` is how right each candidate was (sums to 1)."""
        s = self.scores(enc, consolidated=False)
        p = np.exp(s - s.max())
        p /= p.sum()
        err = np.asarray(target, np.float64) - p
        self._apply(enc, err)
        hit = float(target[int(np.argmax(s))] > 0)
        self.expected += 0.01 * (hit - self.expected)
        return hit

    def reinforce(self, enc, chosen, reward):
        """Dopamine from a scalar outcome only: reward minus what was expected, credited to what was chosen."""
        s = self.scores(enc, consolidated=False)
        p = np.exp(s - s.max())
        p /= p.sum()
        rpe = reward - self.expected
        err = -rpe * p
        err[chosen] += rpe
        self._apply(enc, err)
        self.expected += 0.05 * rpe
        return rpe

    # ------------------------------------------------------------- sleep
    def sleep(self, downscale=0.02, floor=1e-4):
        """Write the day's average into the synapses, shrink everything a little, remove what is left near zero."""
        w = self.consolidated() * (1.0 - downscale)
        pruned = int(((np.abs(w) < floor) & (w != 0)).sum())
        w[np.abs(w) < floor] = 0.0
        self.w, self.u, self.t = w, np.zeros_like(w), 1
        self._cache = None
        live = int((w != 0).sum())
        self.history.append({"trials": self.trials, "synapses": live, "pruned": pruned})
        return {"synapses": live, "pruned": pruned}

    # ------------------------------------------------------------- state
    def synapses(self):
        return int((self.consolidated() != 0).sum())

    def state(self):
        w = self.consolidated()
        nz = np.nonzero(w)[0]
        return {"bits": self.bits, "lr": self.lr, "name": self.name, "trials": self.trials,
                "expected": self.expected, "history": self.history,
                "ids": nz.astype(np.int64), "w": w[nz].astype(np.float32)}

    @classmethod
    def from_state(cls, d):
        s = cls(bits=d["bits"], lr=d["lr"], name=d.get("name", "striatum"))
        s.w[d["ids"]] = d["w"]
        s.trials, s.expected, s.history = d["trials"], d["expected"], d["history"]
        return s

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.state(), f, protocol=4)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            return cls.from_state(pickle.load(f))

In [ ]:
%%writefile /kaggle/working/brain/brain/meaning.py
"""What words mean, learned from the company they keep - by a Hebbian rule, not by gradient descent.

Every word is given a fixed, sparse, random "index" pattern when it is first met: a handful of cells out of a
few hundred, some excited and some inhibited. That is the wiring and it never changes. A word's *meaning*
pattern is then whatever the index patterns of its neighbours have added into it, every time they occurred
together - a synapse from each context word's cells to the word, strengthened by co-occurrence. Two words that
keep the same company end up with the same meaning pattern, which is the distributional hypothesis (Harris,
Firth) implemented as random indexing (Kanerva, Kristoferson & Holst 2000; Sahlgren 2005).

It is weaker than word2vec and it is not trying to be stronger; it is what a plastic network can learn in one
pass with a local rule. It is used for two things: letting "largest" in a question find "biggest" in a passage,
and letting the brain paraphrase its own practice questions so they are less like the sentence they came from.

Frequent words are poor evidence about meaning ("the" is next to everything), so each context word's
contribution is scaled down by how common it is - the same job subsampling does in word2vec, and the same job
a synapse that saturates does in tissue.
"""
import collections
import math
import pickle
import zlib

import numpy as np

from brain.reading import STOP, tokens


class Meaning:
    def __init__(self, dim=256, active=8, window=3, vocab=40000, min_count=3):
        self.dim, self.active, self.window = dim, active, window
        self.max_vocab, self.min_count = vocab, min_count
        self.counts = collections.Counter()
        self.words = []
        self.index = {}
        self.M = None             # meaning patterns, one row per word
        self.unit = None          # the same, normalised
        self._near = {}

    def _index_pattern(self, word):
        """The fixed random pattern a word drives: `active` cells, half excitatory, half inhibitory."""
        g = np.random.default_rng(zlib.crc32(word.encode("utf-8")))
        cells = g.choice(self.dim, self.active, replace=False)
        signs = np.where(np.arange(self.active) % 2 == 0, 1.0, -1.0)
        return cells, signs

    @staticmethod
    def _words(text):
        return [w.lower() for w, _, _ in tokens(text) if w[0].isalpha()]

    def read(self, texts):
        """One pass of Hebbian accumulation over a body of text."""
        docs = [self._words(t) for t in texts]
        for d in docs:
            self.counts.update(d)
        keep = [w for w, n in self.counts.most_common() if n >= self.min_count and w not in STOP]
        keep = keep[: self.max_vocab]
        old = dict(self.index)
        self.words = keep
        self.index = {w: i for i, w in enumerate(keep)}
        M = np.zeros((len(keep), self.dim), np.float32)
        if self.M is not None:
            for w, i in old.items():
                if w in self.index:
                    M[self.index[w]] = self.M[i]
        total = sum(self.counts.values()) or 1
        patterns = {}
        for d in docs:
            ids = [self.index.get(w, -1) for w in d]
            for pos, i in enumerate(ids):
                if i < 0:
                    continue
                lo, hi = max(0, pos - self.window), min(len(d), pos + self.window + 1)
                for q in range(lo, hi):
                    if q == pos or d[q] in STOP:
                        continue
                    c = d[q]
                    if c not in patterns:
                        patterns[c] = self._index_pattern(c)
                    cells, signs = patterns[c]
                    freq = self.counts[c] / total
                    M[i, cells] += signs * min(1.0, math.sqrt(1e-4 / freq))
        self.M = M
        norm = np.linalg.norm(M, axis=1, keepdims=True)
        norm[norm == 0] = 1.0
        self.unit = M / norm
        self._near = {}
        return len(keep)

    def vector(self, word):
        if self.unit is None:
            return None
        i = self.index.get(word.lower())
        return None if i is None else self.unit[i]

    def similarity(self, a, b):
        va, vb = self.vector(a), self.vector(b)
        if va is None or vb is None:
            return 0.0
        return float(va @ vb)

    def near(self, word, top=5, floor=0.35):
        word = word.lower()
        if self.unit is None:
            return []
        if word in self._near:
            return self._near[word]
        i = self.index.get(word)
        if i is None or self.counts[word] < 10:
            self._near[word] = []
            return []
        sims = self.unit @ self.unit[i]
        sims[i] = -1
        best = np.argpartition(-sims, top)[:top]
        best = best[np.argsort(-sims[best])]
        out = [(self.words[k], float(sims[k])) for k in best if sims[k] >= floor]
        self._near[word] = out
        return out

    def neighbour(self, word, rng):
        options = self.near(word)
        return rng.choice(options)[0] if options else None

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump({"dim": self.dim, "active": self.active, "window": self.window, "words": self.words,
                         "M": self.M, "counts": dict(self.counts)}, f, protocol=4)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        m = cls(dim=d["dim"], active=d["active"], window=d["window"])
        m.words, m.M = d["words"], d["M"]
        m.counts = collections.Counter(d["counts"])
        m.index = {w: i for i, w in enumerate(m.words)}
        if m.M is not None:                     # a brain that has not read anything yet has no meanings
            norm = np.linalg.norm(m.M, axis=1, keepdims=True)
            norm[norm == 0] = 1.0
            m.unit = m.M / norm
        return m

In [ ]:
%%writefile /kaggle/working/brain/brain/cloze.py
"""Questions it asks itself about what it has read, with the answer known because it came from the text.

This is where the self in self-training comes from. A teacher's questions are scarce; text is not. Every
sentence it reads can be turned into a question by hiding a stretch of it - a name, a date, a number - and the
hidden stretch is then a right answer it can check itself against. Practising on those is the brain doing what
hippocampal replay is thought to do during rest: re-presenting what was experienced so that the slower systems
learn from it (and what DQN's experience replay borrowed from it).

A question made by blanking a word out of a sentence is too easy and too unlike a real one: every word around
the blank is in the question, so the reader learns that an answer always sits between two question words. Real
questions paraphrase, drop words and put the wh-word in front. So the made-up question is roughed up the same
way - only a clause around the blank is kept, the wh-word is moved to the front, some words are dropped, and
some are swapped for words this brain has learned mean nearly the same thing. That is the "noisy cloze" of
Lewis, Denoyer & Riedel (2019, Unsupervised QA by Cloze Translation), done with nothing but what it has read.
"""
import random
import re

from brain.reading import Passage, STOP

MONTHS = {"january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"}
PLACE_BEFORE = {"in", "at", "near", "from", "to", "across", "into", "throughout"}
YEAR = re.compile(r"^(1[0-9]{3}|20[0-9]{2})s?$")


def _answers(p, a, b):
    """Stretches of one sentence worth asking about, with the wh-word that would ask for each."""
    out = []
    i = a + 1                         # the first word of a sentence is capitalised for no reason
    while i < b:
        w = p.words[i]
        if w[0].isdigit() or (p.low[i] in MONTHS and i + 1 < b and p.words[i + 1][0].isdigit()):
            j = i + 1
            # a date is one answer however it is written: "22 April 2008", "April 22, 2008", "1850s"
            while j < b and (p.words[j][0].isdigit() or p.low[j] in MONTHS or p.low[j] in (
                    "million", "billion", "thousand", "percent", "%") or (p.words[j] == "," and j + 1 < b
                                                                         and p.words[j + 1][0].isdigit())):
                j += 1
            span = p.low[i:j]
            prev = p.low[i - 1]
            if any(YEAR.match(x) for x in span) or any(x in MONTHS for x in span) or prev in MONTHS:
                wh = "when"
            else:
                wh = "how many" if j < b and p.words[j][0].islower() else "what"
            out.append((i, j, wh))
            i = j
            continue
        if w[0].isupper() and w.lower() not in STOP:
            j = i + 1
            while j < b and (p.words[j][0].isupper() or (p.low[j] in ("of", "the", "de", "and")
                                                         and j + 1 < b and p.words[j + 1][0].isupper())):
                j += 1
            prev = p.low[i - 1] if i > a else ""
            wh = "where" if prev in PLACE_BEFORE else ("who" if j < b and p.low[j] in (
                "was", "is", "who", "became", "said", "wrote", "founded", "led", "won") else "what")
            out.append((i, j, wh))
            i = j
            continue
        i += 1
    return out


def make(passage_text, rng, meaning=None, per_sentence=1, drop=0.15, swap=0.2, window=9):
    """-> [(question, answer_text)] asked of this passage by itself."""
    p = Passage(passage_text)
    out = []
    for a, b in p.sentences:
        if b - a < 6:
            continue
        found = _answers(p, a, b)
        rng.shuffle(found)
        for i, j, wh in found[:per_sentence]:
            lo, hi = max(a, i - window), min(b, j + window)
            words = []
            for k in list(range(lo, i)) + list(range(j, hi)):
                w = p.words[k]
                if not w[0].isalnum() and (w not in ",;:" or rng.random() < 0.7):
                    continue                                   # quotes, brackets, most commas
                low = w.lower()
                if low not in STOP and rng.random() < drop:
                    continue                                   # questions leave things out
                if meaning is not None and low not in STOP and w[0].islower() and rng.random() < swap:
                    alt = meaning.neighbour(low, rng)
                    if alt:
                        w = alt                                # and say them differently
                words.append(w)
            if len(words) < 3:
                continue
            answer = p.text[p.toks[i][1]:p.toks[j - 1][2]]
            q = wh.capitalize() + " " + " ".join(words) + "?"
            out.append((q, answer))
    return out

In [ ]:
%%writefile /kaggle/working/brain/brain/conversation.py
"""Talking: what people said after something like this, and which of those fits now.

A person's first reply to "how is your day?" is not composed from a grammar; it is recalled - they have heard
that exchange a thousand times. So the conversational memory stores exchanges as episodes (what was said, what
was said back), recalls the ones whose opening resembles what was just said, and the striatum chooses among the
replies they offer. Nothing about how to reply is written here: which reply fits is learned by practice, with
the reply that was actually given as the outcome.

Three things describe how well a reply fits, all learned from what was heard:

  recall      how similar the moment is to moments this reply (or one like it) followed before - hippocampal
              episodic recall, the kNN-LM idea (Khandelwal et al. 2020) applied to turns instead of words.
  association the Hebbian link between words across a turn boundary: "thanks" tends to be answered by
              "welcome", "how are you" by "fine". Counted, and scored as pointwise mutual information.
  overlap     plain shared words - the baseline every retrieval dialogue system has to beat (TF-IDF, Lowe et
              al. 2015).

The striatum learns how to weigh them, and a few thousand lexical pairs across the boundary besides, from
practice: shown a moment and ten candidate replies, one of which was really said.
"""
import collections
import math
import re

import numpy as np
import scipy.sparse as sp

WORD = re.compile(r"[a-z0-9']+|[?!.]")
QUOTED = re.compile(r"'([^']*)'|\"([^\"]*)\"")


def words(text):
    return WORD.findall(text.lower())


def parse_dailydialog(path):
    """DailyDialog as Kaggle ships it: one dialogue per row, turns as a numpy-style list of strings."""
    import csv
    csv.field_size_limit(1 << 30)
    out = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            turns = [" ".join((a or b).split()) for a, b in QUOTED.findall(row["dialog"])]
            turns = [t for t in turns if t]
            if len(turns) >= 2:
                out.append(turns)
    return out


class Vectoriser:
    """Words -> sparse, rarity-weighted vectors. The rarity is counted from what was heard."""

    def __init__(self):
        self.vocab = {}
        self.df = None
        self.n = 0

    def fit(self, texts):
        df = collections.Counter()
        for t in texts:
            df.update(set(words(t)))
        self.vocab = {w: i for i, (w, c) in enumerate(df.most_common()) if c >= 2}
        self.df = np.zeros(len(self.vocab))
        for w, i in self.vocab.items():
            self.df[i] = df[w]
        self.n = len(texts)
        self.idf = np.log((self.n + 1) / (self.df + 1)) + 1.0
        return self

    def __call__(self, texts, weighted=True, binary=False):
        rows, cols, vals = [], [], []
        for r, t in enumerate(texts):
            c = collections.Counter(w for w in words(t) if w in self.vocab)
            for w, k in c.items():
                i = self.vocab[w]
                rows.append(r)
                cols.append(i)
                v = 1.0 if binary else (1 + math.log(k))
                vals.append(v * (self.idf[i] if weighted else 1.0))
        m = sp.csr_matrix((vals, (rows, cols)), shape=(len(texts), len(self.vocab)))
        if weighted:
            norm = np.sqrt(np.asarray(m.multiply(m).sum(1)).ravel())
            norm[norm == 0] = 1.0
            m = sp.diags(1.0 / norm) @ m
        return m.tocsr()


class Conversation:
    def __init__(self):
        self.contexts, self.replies = [], []
        self.vec = None

    # ------------------------------------------------------------------ hearing
    def hear(self, dialogues):
        """Store every exchange in these dialogues as an episode, and count what answered what."""
        for turns in dialogues:
            for a, b in zip(turns, turns[1:]):
                self.contexts.append(a)
                self.replies.append(b)
        self.vec = Vectoriser().fit(self.contexts + self.replies)
        self.C = self.vec(self.contexts)
        self.R = self.vec(self.replies)
        # association across the turn boundary: co-occurrence of a word before it and a word after it
        Xb = self.vec(self.contexts, weighted=False, binary=True)
        Yb = self.vec(self.replies, weighted=False, binary=True)
        joint = (Xb.T @ Yb).tocoo()
        n = len(self.contexts)
        px = np.asarray(Xb.sum(0)).ravel() / n
        py = np.asarray(Yb.sum(0)).ravel() / n
        keep = joint.data >= 3
        r, c, v = joint.row[keep], joint.col[keep], joint.data[keep] / n
        pmi = np.log(v / (px[r] * py[c]))
        pmi = np.maximum(pmi, 0.0)
        self.PMI = sp.csr_matrix((pmi, (r, c)), shape=(len(self.vec.vocab),) * 2)
        return len(self.contexts)

    # --------------------------------------------------------------- describing
    def recall(self, context, k=30):
        """The stored moments most like this one: (similarity, index)."""
        q = self.vec([context])
        sims = (self.C @ q.T).toarray().ravel()
        top = np.argpartition(-sims, min(k, len(sims) - 1))[:k]
        top = top[np.argsort(-sims[top])]
        return [(float(sims[i]), int(i)) for i in top if sims[i] > 0]

    def describe(self, context, candidates, recalled=None):
        """Features of each candidate reply to this context."""
        q = self.vec([context])
        R = self.vec(candidates)
        overlap = (R @ q.T).toarray().ravel()
        qb = self.vec([context], weighted=False, binary=True)
        Rb = self.vec(candidates, weighted=False, binary=True)
        assoc_raw = np.asarray((Rb @ (self.PMI.T @ qb.T)).todense()).ravel()
        nq = max(1, qb.nnz)
        nr = np.maximum(1, np.diff(Rb.indptr))
        assoc = assoc_raw / np.sqrt(nq * nr)
        recalled = recalled if recalled is not None else self.recall(context)
        if recalled:
            sims = np.array([s for s, _ in recalled])
            past = self.R[[i for _, i in recalled]]
            match = (R @ past.T).toarray()                      # candidate x recalled reply
            knn = (match * sims[None, :]).max(1)
            knn_sum = (match * sims[None, :]).sum(1) / (sims.sum() or 1.0)
        else:
            knn = knn_sum = np.zeros(len(candidates))
        cw = words(context)
        asks = "?" in cw
        c_first = cw[0] if cw else "<none>"
        c_last = [w for w in cw if w not in "?!."][-3:]
        out = []
        for k, cand in enumerate(candidates):
            rw = words(cand)
            r_first = rw[0] if rw else "<none>"
            r_open = rw[:3]
            L = len(rw)
            f = [("overlap", overlap[k]), ("assoc", assoc[k]), ("knn", knn[k]), ("knn_sum", knn_sum[k]),
                 (f"assoc|{asks}", assoc[k]), (f"knn|{asks}", knn[k]),
                 f"b|{asks}", f"rfirst|{asks}|{r_first}", f"pair|{c_first}|{r_first}",
                 f"rlen|{asks}|{min(L // 4, 6)}", f"rq|{asks}|{'?' in rw}",
                 f"knnb|{int(min(knn[k], 0.99) * 10)}", f"assocb|{int(min(assoc[k], 1.99) * 5)}"]
            f += [f"x|{x}|{y}" for x in c_last for y in r_open]
            out.append(f)
        return out

    # ------------------------------------------------------------------ acting
    def practise(self, striatum, context, reply, distractors, rng=None):
        cands = [reply] + list(distractors)
        order = list(range(len(cands)))
        (rng or np.random.default_rng()).shuffle(order)
        cands = [cands[i] for i in order]
        gold = order.index(0)
        enc = striatum.encode(self.describe(context, cands, self._recall_excluding(context, reply)))
        target = np.zeros(len(cands))
        target[gold] = 1.0
        s = striatum.scores(enc, consolidated=False)
        top = np.flatnonzero(s == s.max())
        hit = int(top[0]) == gold and len(top) == 1
        striatum.teach(enc, target)
        return hit

    def _recall_excluding(self, context, reply, k=30):
        """Recall without the episode being practised on - remembering the answer is not knowing it."""
        got = self.recall(context, k + 2)
        return [(s, i) for s, i in got if not (self.replies[i] == reply and self.contexts[i] == context)][:k]

    def choose(self, striatum, context, candidates, recalled=None):
        enc = striatum.encode(self.describe(context, candidates, recalled))
        k, s = striatum.choose(enc)
        return k, s

    def reply(self, striatum, context, pool=40):
        """Say something: the replies that followed moments like this one, and the striatum's pick of them."""
        recalled = self.recall(context, pool)
        if not recalled:
            return None, 0.0
        seen, cands = set(), []
        for _, i in recalled:
            r = self.replies[i]
            if r not in seen:
                seen.add(r)
                cands.append(r)
        k, s = self.choose(striatum, context, cands, recalled)
        return cands[k], float(s[k])

In [ ]:
%%writefile /kaggle/working/brain/brain/selftrain.py
"""The brain training itself: read, practise, sleep, sit an exam, keep what it learned, and go round again.

    python brain/selftrain.py experiment --seeds 5 --out results/selftrain      # the pre-registered test
    python brain/selftrain.py loop --hours 9 --state state/ [--resume old_state/] # the self-training loop

One cycle is a day:

  wake      read pages chosen by curiosity - links out of what it has already read, weighted toward the pages
            its own questions went worst on (active sampling toward what it does not yet understand; Gottlieb
            & Oudeyer 2018). Each passage is stored in the hippocampus once, with a priority that is higher
            the more of it was new.
  practise  three kinds of trial, each ending in a dopamine signal to a striatum:
              teacher   a real reading-comprehension question with its answer (SQuAD train), a few thousand a day
              self      questions it makes itself out of what it read, answer known from the text (cloze.py)
              talk      a real exchange from a conversation, with the reply that was actually given
            Self-practice replays stored passages by priority - the worst-understood and newest first, which
            is prioritised replay (Mattar & Daw 2018) rather than a uniform sweep.
  sleep     both striata consolidate: the day's running average is written into the synapses, everything is
            downscaled a little and what is left near zero is pruned. Word meanings are re-accumulated from
            the day's reading.
  exam      frozen weights, held-out questions and held-out conversations it has never practised on: SQuAD
            dev (F1, EM) and DailyDialog test (choose the real reply of ten, R@1). Nothing is ever tuned on
            these. The result is appended to the log and the whole brain is checkpointed, so the next run
            resumes where this one stopped.

The teacher's questions run out after about thirty days. After that it is self-practice and reading alone,
and whether the exam keeps moving then is the most honest measure of whether it is teaching itself.
"""
import argparse
import collections
import glob
import json
import math
import os
import pickle
import random
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path[:0] = [ROOT, os.path.join(ROOT, "chat")]

import numpy as np  # noqa: E402

from brain import cloze  # noqa: E402
from brain.conversation import Conversation, parse_dailydialog  # noqa: E402
from brain.meaning import Meaning  # noqa: E402
from brain.reading import Passage, Question, Rarity, Reader, best, exact, f1  # noqa: E402
from brain.striatum import Striatum  # noqa: E402

CORPORA = ("/kaggle/input", "E:/corpora", os.path.join(os.path.expanduser("~"), "flybrain", "corpora"))
PUBLISHED_SLIDING_WINDOW_F1 = 0.202       # SQuAD v1.1 dev, Rajpurkar et al. (2016) table 2

SEEDS = ["Science", "History", "Mathematics", "Biology", "Physics", "Chemistry", "Geography", "Music",
         "Literature", "Philosophy", "Economics", "Medicine", "Technology", "Art", "Religion", "Sport",
         "Language", "Politics", "Astronomy", "Computer science", "Psychology", "Law", "Architecture", "Film",
         "Agriculture", "Ancient Rome", "World War II", "Industrial Revolution", "Evolution", "Climate"]


# ---------------------------------------------------------------------------------------------- data
def find(name):
    for root in CORPORA:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name)


def squad(path):
    out = []
    for a in json.load(open(path, encoding="utf-8"))["data"]:
        for p in a["paragraphs"]:
            for qa in p["qas"]:
                out.append((a["title"], p["context"], qa["question"], [x["text"] for x in qa["answers"]]))
    return out


def load_data():
    train = squad(find("train-v1.1.json"))
    dev = squad(find("dev-v1.1.json"))
    dtrain = parse_dailydialog(_dd("train.csv"))
    dtest = parse_dailydialog(_dd("test.csv"))
    return train, dev, dtrain, dtest


def _dd(name):
    for root in CORPORA:
        for hit in glob.glob(os.path.join(root, "**", name), recursive=True):
            if "dialog" in hit.lower():
                return hit
    raise FileNotFoundError(name)


def dialogue_exam_items(dtest, seed=7, limit=None):
    """Fixed test items: the real reply and nine others from the test set, in a fixed shuffled order.

    The real reply must not sit in a fixed place. With it always first, anything that scores every candidate
    the same - an untrained striatum, a word-overlap score of zero for all ten - "chose" it every time.
    """
    pairs = [(a, b) for t in dtest for a, b in zip(t, t[1:])]
    rng = random.Random(seed)
    replies = [b for _, b in pairs]
    items = []
    distinct = len(set(replies))
    for a, b in pairs:
        ds = []
        want = min(9, distinct - 1)
        while len(ds) < want:
            d = rng.choice(replies)
            if d != b and d not in ds:
                ds.append(d)
        cands = [b] + ds
        rng.shuffle(cands)
        items.append((a, cands, cands.index(b)))
    return items[:limit] if limit else items


def pick(values, rng):
    """argmax with ties broken at random, so a flat score cannot look like a choice."""
    values = np.asarray(values, float)
    top = np.flatnonzero(values == values.max())
    return int(rng.choice(top)) if len(top) > 1 else int(top[0])


# ---------------------------------------------------------------------------------------------- the brain
class Brain:
    def __init__(self):
        self.rarity = Rarity()
        self.meaning = Meaning()
        self.reader = Reader(self.rarity)
        self.qa = Striatum(bits=22, lr=0.1, name="reading")
        self.talk = Striatum(bits=20, lr=0.05, name="talking")
        self.conv = Conversation()
        self.passages = []            # hippocampus: {"text", "source", "cycle", "novelty", "quiz"}
        self.read = set()
        self.frontier = collections.Counter()
        self.links = {}               # page -> the pages it links to, so a bad quiz can send it back to read
        self.log = []
        self.day = 0
        self.teacher_seen = 0
        self.talk_seen = 0
        self.best_val, self.best_qa = -1.0, None      # the best night so far, on questions set aside

    # ---- memory
    def store(self, text, source):
        stems = [s for s in Passage(text).stems if s[:1].isalnum()]
        novelty = sum(1 for s in set(stems) if self.rarity.df.get(s, 0) < 2) / max(1, len(set(stems)))
        self.rarity.read(text)
        self.passages.append({"text": text, "source": source, "cycle": self.day, "novelty": novelty,
                              "quiz": None})

    def replay_order(self, n, rng):
        """Which stored passages to practise on tonight: newest, least understood and most novel first."""
        if not self.passages:
            return []
        pr = np.array([(1.0 - (p["quiz"] if p["quiz"] is not None else 0.0)) + p["novelty"]
                       + (1.0 if p["cycle"] == self.day else 0.0) for p in self.passages]) + 1e-3
        k = min(n, len(self.passages))
        idx = np.random.default_rng(rng.randrange(1 << 30)).choice(len(self.passages), size=k, replace=False,
                                                                 p=pr / pr.sum())
        return [int(i) for i in idx]

    # ---- persistence
    def save(self, d):
        os.makedirs(d, exist_ok=True)
        self.qa.save(os.path.join(d, "reading_striatum.pkl"))
        self.talk.save(os.path.join(d, "talking_striatum.pkl"))
        self.meaning.save(os.path.join(d, "meaning.pkl"))
        with open(os.path.join(d, "brain.pkl"), "wb") as f:
            pickle.dump({"rarity": (self.rarity.docs, dict(self.rarity.df)), "passages": self.passages,
                         "read": sorted(self.read), "frontier": dict(self.frontier), "links": self.links,
                         "day": self.day,
                         "teacher_seen": self.teacher_seen, "talk_seen": self.talk_seen,
                         "best_val": self.best_val, "best_qa": self.best_qa}, f, protocol=4)
        json.dump(self.log, open(os.path.join(d, "log.json"), "w"), indent=1)

    @classmethod
    def load(cls, d, dtrain):
        b = cls()
        b.qa = Striatum.load(os.path.join(d, "reading_striatum.pkl"))
        b.talk = Striatum.load(os.path.join(d, "talking_striatum.pkl"))
        b.meaning = Meaning.load(os.path.join(d, "meaning.pkl"))
        with open(os.path.join(d, "brain.pkl"), "rb") as f:
            s = pickle.load(f)
        b.rarity.docs, b.rarity.df = s["rarity"][0], collections.Counter(s["rarity"][1])
        b.reader = Reader(b.rarity)
        b.passages, b.read, b.frontier = s["passages"], set(s["read"]), collections.Counter(s["frontier"])
        b.links = s.get("links", {})
        b.day, b.teacher_seen, b.talk_seen = s["day"], s["teacher_seen"], s["talk_seen"]
        b.best_val, b.best_qa = s.get("best_val", -1.0), s.get("best_qa")
        b.log = json.load(open(os.path.join(d, "log.json")))
        b.conv.hear(dtrain)
        return b


# ---------------------------------------------------------------------------------------------- exams
def exam_reading(brain, dev, limit=None):
    items = dev[:limit] if limit else dev
    F = E = 0.0
    by = collections.defaultdict(lambda: [0.0, 0])
    for _, c, q, g in items:
        a, _, _ = brain.reader.answer(brain.qa, q, c)
        s = best(a, g, f1)
        F += s
        E += best(a, g, exact)
        wh = Question(q, brain.rarity).wh
        by[wh][0] += s
        by[wh][1] += 1
    n = max(1, len(items))
    return {"f1": F / n, "em": E / n, "n": len(items),
            "by_wh": {k: round(v[0] / v[1], 3) for k, v in by.items() if v[1] >= 30}}


def exam_talking(brain, items):
    hits = 0
    rng = random.Random(11)
    for a, cands, gold in items:
        _, s = brain.conv.choose(brain.talk, a, cands)
        hits += pick(s, rng) == gold
    return {"r_at_1": hits / max(1, len(items)), "n": len(items)}


def sliding_window(brain, dev):
    """Richardson et al. (2013)'s baseline as the SQuAD paper ran it, over this reader's own candidates:
    the span whose surroundings share the most (rarity-weighted) words with the question."""
    F = E = 0.0
    for _, c, q, g in dev:
        qq, p = Question(q, brain.rarity), Passage(c)
        cands = brain.reader.candidates(qq, p)
        if not cands:
            continue
        width = max(3, len(qq.stems))
        best_s, pick = -1.0, None
        for i, j, _ in cands:
            lo, hi = max(0, i - width), min(len(p.stems), j + width)
            got = {p.stems[k] for k in range(lo, hi) if not (i <= k < j)} & qq.stems
            s = sum(qq.weight[x] for x in got) - 0.01 * (j - i)
            if s > best_s:
                best_s, pick = s, (i, j)
        a = p.text[p.toks[pick[0]][1]:p.toks[pick[1] - 1][2]]
        F += best(a, g, f1)
        E += best(a, g, exact)
    return {"f1": F / len(dev), "em": E / len(dev)}


# ---------------------------------------------------------------------------------------------- one arm
def practise_reading(brain, teacher, selfq, sleep=True):
    for c, q, g in selfq:
        brain.reader.practise(brain.qa, q, c, g)
    if selfq and sleep:
        brain.qa.sleep(downscale=0.0)
    for _, c, q, g in teacher:
        brain.reader.practise(brain.qa, q, c, g)
    if teacher and sleep:
        brain.qa.sleep(downscale=0.0)


def run_arm(args):
    """One arm of the pre-registered experiment, one seed. Everything it needs is passed or global."""
    name, seed, teacher_n, self_n, swap = args
    G = _G
    brain = Brain()
    brain.rarity, brain.reader, brain.meaning = G["rarity"], Reader(G["rarity"]), G["meaning"]
    rng = random.Random(seed)
    order = list(range(len(G["train"])))
    rng.shuffle(order)
    teacher = [G["train"][i] for i in order[:teacher_n]]
    taught = {t[1] for t in teacher}
    pool = [c for c in G["contexts"] if c not in taught]
    rng.shuffle(pool)
    selfq = []
    for c in pool:
        if len(selfq) >= self_n:
            break
        for q, a in cloze.make(c, rng, G["meaning"] if swap > 0 else None, swap=swap):
            selfq.append((c, q, [a]))
    selfq = selfq[:self_n]
    t0 = time.time()
    practise_reading(brain, teacher, selfq)
    r = exam_reading(brain, G["dev"])
    return {"arm": name, "seed": seed, "teacher": teacher_n, "self": len(selfq), "swap": swap,
            "f1": r["f1"], "em": r["em"], "by_wh": r["by_wh"], "seconds": round(time.time() - t0)}


def run_talk_arm(args):
    seed, n = args
    G = _G
    brain = Brain()
    brain.conv = G["conv"]
    rng = random.Random(seed)
    pairs = list(G["dpairs"])
    rng.shuffle(pairs)
    replies = [b for _, b in G["dpairs"]]
    for a, b in pairs[:n]:
        ds = []
        while len(ds) < 9:
            d = rng.choice(replies)
            if d != b:
                ds.append(d)
        brain.conv.practise(brain.talk, a, b, ds, rng)
    brain.talk.sleep(downscale=0.0)
    return {"arm": "talk", "seed": seed, "practised": n, **exam_talking(brain, G["ditems"])}


_G = {}


def experiment(out, seeds=5, workers=4, dev_limit=None, teacher_n=1000, self_n=20000, talk_n=20000):
    """The pre-registered test (PREREGISTRATION_selftrain.md). Arms x seeds, in parallel."""
    import multiprocessing as mp
    os.makedirs(out, exist_ok=True)
    train, dev, dtrain, dtest = load_data()
    dev = dev[:dev_limit] if dev_limit else dev
    contexts = sorted({c for _, c, _, _ in train})
    rarity = Rarity()
    for c in contexts:
        rarity.read(c)
    meaning = Meaning()
    meaning.read(contexts)
    conv = Conversation()
    conv.hear(dtrain)
    _G.update(train=train, dev=dev, contexts=contexts, rarity=rarity, meaning=meaning, conv=conv,
              dpairs=[(a, b) for t in dtrain for a, b in zip(t, t[1:])],
              ditems=dialogue_exam_items(dtest))
    probe = Brain()
    probe.rarity, probe.reader, probe.conv = rarity, Reader(rarity), conv
    baselines = {"sliding_window": sliding_window(probe, dev)}
    for key in ("overlap", "assoc", "knn"):
        hits = 0
        rng = random.Random(11)
        for a, cands, gold in _G["ditems"]:
            feats = conv.describe(a, cands)
            vals = [dict(x for x in fs if isinstance(x, tuple))[key] for fs in feats]
            hits += pick(vals, rng) == gold
        baselines[f"talk_{key}"] = hits / len(_G["ditems"])
    print("baselines:", json.dumps(baselines), flush=True)
    arms = []
    for s in range(seeds):
        arms += [("teacher", s, teacher_n, 0, 0.0), ("teacher+self", s, teacher_n, self_n, 0.2),
                 ("self", s, 0, self_n, 0.2), ("teacher+self-noswap", s, teacher_n, self_n, 0.0)]
    ctx = mp.get_context("fork") if hasattr(os, "fork") else mp.get_context("spawn")
    results = []
    if workers > 1 and hasattr(os, "fork"):
        with ctx.Pool(workers) as pool:
            for r in pool.imap_unordered(run_arm, arms):
                print(json.dumps(r), flush=True)
                results.append(r)
            for r in pool.imap_unordered(run_talk_arm, [(s, talk_n) for s in range(seeds)]):
                print(json.dumps(r), flush=True)
                results.append(r)
    else:
        for a in arms:
            r = run_arm(a)
            print(json.dumps(r), flush=True)
            results.append(r)
        for s in range(seeds):
            r = run_talk_arm((s, talk_n))
            print(json.dumps(r), flush=True)
            results.append(r)
    json.dump({"baselines": baselines, "results": results}, open(os.path.join(out, "experiment.json"), "w"),
              indent=1)
    return baselines, results


def analyse(baselines, results, alpha=0.05):
    """The four pre-registered tests, Holm-Bonferroni across them."""
    from scipy import stats
    by = collections.defaultdict(dict)
    for r in results:
        by[r["arm"]][r["seed"]] = r
    seeds = sorted(by["teacher"])

    def paired(a, b, key="f1"):
        x = np.array([by[a][s][key] for s in seeds])
        y = np.array([by[b][s][key] for s in seeds])
        t, p = stats.ttest_rel(x, y)
        return float((x - y).mean()), float(t), float(p)

    def against(a, value, key):
        x = np.array([by[a][s][key] for s in sorted(by[a])])
        t, p = stats.ttest_1samp(x, value)
        return float(x.mean() - value), float(t), float(p)

    tests = [("H1 teacher+self > teacher (F1)",) + paired("teacher+self", "teacher"),
             # against whichever sliding window is stronger: ours over our candidates, or the published dev
             # figure (Rajpurkar et al. 2016, 20.2 F1) - a baseline weakened by reimplementation proves nothing
             ("H2 self alone > sliding window (F1)",) + against("self", max(baselines["sliding_window"]["f1"],
                                                                           PUBLISHED_SLIDING_WINDOW_F1), "f1"),
             ("H3 paraphrase helps: teacher+self > noswap (F1)",) + paired("teacher+self", "teacher+self-noswap"),
             ("H4 learned talk > best single signal (R@1)",) + against("talk", max(
                 baselines["talk_overlap"], baselines["talk_assoc"], baselines["talk_knn"]), "r_at_1")]
    order = sorted(range(len(tests)), key=lambda i: tests[i][3])
    verdict, alive = {}, True
    for rank, i in enumerate(order):
        name, diff, t, p = tests[i]
        # Holm step-down: keep going while nulls are rejected. A hypothesis is supported when its null is rejected
        # *in the predicted direction*; a rejection the wrong way is not support, but it does not stop the
        # procedure either (the first version stopped there, which is not Holm and cost the maturation run H1/H3)
        alive = alive and p <= alpha / (len(tests) - rank)
        verdict[name] = "SUPPORTED" if (alive and diff > 0) else (
            "rejected, opposite direction" if alive else "not supported")
    means = {arm: {k: round(float(np.mean([r[k] for r in rs.values()])), 4) for k in ("f1", "em", "r_at_1")
                   if k in next(iter(rs.values()))} for arm, rs in by.items()}
    return {"tests": [dict(zip(("name", "diff", "t", "p"), x)) for x in tests], "verdict": verdict,
            "means": means, "baselines": baselines}


# ---------------------------------------------------------------------------------------------- the loop
def wake(brain, pages, rng, verbose=True):
    """Read what curiosity picks. Returns the titles read."""
    import web
    got = []
    if not brain.frontier:
        for s in SEEDS:
            brain.frontier[s] += 1
    tries = 0
    while len(got) < pages and brain.frontier and tries < pages * 4:
        tries += 1
        # curiosity is not greedy: mostly the most-linked unread page, sometimes a less obvious one
        top = [t for t, _ in brain.frontier.most_common(40) if t not in brain.read]
        if not top:
            break
        title = top[0] if rng.random() < 0.6 else rng.choice(top)
        del brain.frontier[title]
        brain.read.add(title)
        sents, url, err = web.read_page(title, sentences=400)
        if not sents or len(sents) < 8:
            continue
        for k in range(0, len(sents), 5):
            brain.store(" ".join(sents[k:k + 5]), title)
        try:
            d = web._get({"action": "query", "prop": "links", "titles": title, "pllimit": "max",
                          "plnamespace": 0})
            out = []
            for _, pg in d.get("query", {}).get("pages", {}).items():
                for link in pg.get("links", []):
                    t = link["title"]
                    if not t.startswith(("List of", "Lists of", "Index of", "Outline of")):
                        out.append(t)
                        if t not in brain.read:
                            brain.frontier[t] += 1
            brain.links[title] = out[:200]
        except Exception as e:
            print("   links failed:", type(e).__name__, e, flush=True)
        got.append(title)
    if verbose:
        print(f"   read {len(got)} page(s): {', '.join(got[:8])}{' ...' if len(got) > 8 else ''}", flush=True)
    return got


def day(brain, data, rng, pages=12, teacher_n=3000, self_n=4000, talk_n=3000):
    train, dev, dtrain, dtest, ditems, dpairs, val = data
    brain.day += 1
    t0 = time.time()
    # wake
    new = wake(brain, pages, rng)
    if new:
        brain.meaning.read([p["text"] for p in brain.passages if p["cycle"] == brain.day])
    # practise: the teacher's questions for today
    teacher = train[brain.teacher_seen:brain.teacher_seen + teacher_n]
    brain.teacher_seen += len(teacher)
    # a passage a teacher asked about is a passage it read: it counts toward how rare each word is, the same
    # as anything read by curiosity (the exam's passages never do - the exam leaves the brain as it found it)
    for passage in dict.fromkeys(c for _, c, _, _ in teacher):
        brain.rarity.read(passage)
    t_score = [brain.reader.practise(brain.qa, q, c, g)[0] for _, c, q, g in teacher]
    # rehearsal: once the teacher has nothing new, what it was taught is replayed alongside what it teaches
    # itself. Session 1 without this: reading rose to 46 F1 while there were teacher questions, then self-practice
    # alone pulled it down to 37 in nine days - new learning overwriting old, which is exactly what interleaved
    # replay of old experience prevents (McClelland, McNaughton & O'Reilly 1995)
    rehearsed = 0
    if not teacher and brain.teacher_seen:
        seen = min(brain.teacher_seen, len(train))
        for k in rng.sample(range(seen), min(teacher_n, seen)):
            _, c, q, g = train[k]
            brain.reader.practise(brain.qa, q, c, g)
            rehearsed += 1
    # practise: questions it asks itself, replaying passages by priority
    s_hits, s_n = [], 0
    for i in brain.replay_order(max(1, self_n // 4), rng):
        p = brain.passages[i]
        qs = cloze.make(p["text"], rng, brain.meaning, swap=0.2)
        scores = []
        for q, a in qs:
            sc, ok = brain.reader.practise(brain.qa, q, p["text"], [a])
            if ok:
                scores.append(sc)
        s_n += len(qs)
        if scores:
            p["quiz"] = float(np.mean(scores))
            s_hits += scores
            if p["quiz"] < 0.3:
                # what it understood worst is what it goes and reads around next
                for link in brain.links.get(p["source"], [])[:20]:
                    if link not in brain.read:
                        brain.frontier[link] += 2
        if s_n >= self_n:
            break
    # practise: conversation, the real reply as the outcome
    talk = dpairs[brain.talk_seen % len(dpairs):][:talk_n]
    brain.talk_seen += len(talk)
    replies = [b for _, b in dpairs]
    t_hits = []
    for a, b in talk:
        ds = []
        while len(ds) < 9:
            d = rng.choice(replies)
            if d != b:
                ds.append(d)
        t_hits.append(brain.conv.practise(brain.talk, a, b, ds, rng))
    # sleep
    s1 = brain.qa.sleep(downscale=0.01, floor=1e-4)
    s2 = brain.talk.sleep(downscale=0.01, floor=1e-4)
    # sleep keeps what the day earned and not what it cost. Before the new weights are trusted they are tried on
    # questions set aside from the teacher's own pile (never practised on, and not the exam); if the day made
    # them worse, the reader goes back to its best night so far. Consolidation is selective in the brain as well
    # - not everything a day wrote survives the night.
    checked = exam_reading(brain, val)["f1"]
    kept = checked >= brain.best_val - 0.003
    if kept:
        if checked > brain.best_val:
            brain.best_val, brain.best_qa = checked, brain.qa.state()
    elif brain.best_qa is not None:
        # a worse night is blended in, not thrown away. Session 2 threw every such night away, and because the
        # check's own noise on 2,000 questions (~0.005 F1) was larger than a night's real progress, one lucky
        # night froze reading for thirty days. Half of the night is kept - its learning accumulates across
        # nights while any one bad night is diluted - which is closer to what sleep does anyway: it integrates
        # the day into what was already there rather than accepting or refusing it whole.
        best = Striatum.from_state(brain.best_qa)
        mixed = 0.5 * (brain.qa.consolidated() + best.consolidated())
        brain.qa.w, brain.qa.u, brain.qa.t, brain.qa._cache = mixed, np.zeros_like(mixed), 1, None
    # exam
    reading = exam_reading(brain, dev)
    talking = exam_talking(brain, ditems)
    entry = {"day": brain.day, "seconds": round(time.time() - t0), "pages": new, "passages": len(brain.passages),
             "teacher_seen": brain.teacher_seen, "teacher_today": len(teacher), "rehearsed": rehearsed,
             "validation_f1": round(checked, 4), "kept_day": kept, "best_validation_f1": round(brain.best_val, 4),
             "teacher_practice_f1": round(float(np.mean(t_score)), 4) if t_score else None,
             "self_questions": s_n, "self_practice_f1": round(float(np.mean(s_hits)), 4) if s_hits else None,
             "talk_practice_hit": round(float(np.mean(t_hits)), 4) if t_hits else None,
             "reading_synapses": s1["synapses"], "reading_pruned": s1["pruned"],
             "talking_synapses": s2["synapses"], "talking_pruned": s2["pruned"],
             "exam_f1": round(reading["f1"], 4), "exam_em": round(reading["em"], 4),
             "exam_by_wh": reading["by_wh"], "exam_r_at_1": round(talking["r_at_1"], 4),
             "vocabulary": len(brain.meaning.words)}
    brain.log.append(entry)
    print(f"day {brain.day}: F1 {entry['exam_f1']:.3f} EM {entry['exam_em']:.3f} | talk R@1 "
          f"{entry['exam_r_at_1']:.3f} | teacher {brain.teacher_seen:,} self {s_n} | "
          f"{len(brain.passages):,} passages | {entry['seconds']}s", flush=True)
    return entry


def loop(state, hours, resume=None, pages=12, teacher_n=3000, self_n=4000, talk_n=3000, seed=0):
    train, dev, dtrain, dtest = load_data()
    rng = random.Random(seed)
    # questions set aside for sleep to check itself on: ten whole articles, so no passage is shared with practice
    titles = sorted({t for t, _, _, _ in train})
    held = set(random.Random(99).sample(titles, 10))
    val = [x for x in train if x[0] in held][:2000]
    train = [x for x in train if x[0] not in held]
    order = list(range(len(train)))
    random.Random(1234).shuffle(order)
    train = [train[i] for i in order]
    dpairs = [(a, b) for t in dtrain for a, b in zip(t, t[1:])]
    random.Random(4321).shuffle(dpairs)
    ditems = dialogue_exam_items(dtest)
    if resume and os.path.exists(os.path.join(resume, "brain.pkl")):
        brain = Brain.load(resume, dtrain)
        print(f"resumed at day {brain.day}: {len(brain.passages):,} passages, teacher {brain.teacher_seen:,}",
              flush=True)
    else:
        brain = Brain()
        brain.conv.hear(dtrain)
        # day 0: an exam before any practice, so every later number has something to be compared with
        r, t = exam_reading(brain, dev), exam_talking(brain, ditems)
        brain.log.append({"day": 0, "exam_f1": round(r["f1"], 4), "exam_em": round(r["em"], 4),
                          "exam_r_at_1": round(t["r_at_1"], 4), "teacher_seen": 0, "passages": 0})
        print(f"day 0 (untrained): F1 {r['f1']:.3f} | talk R@1 {t['r_at_1']:.3f}", flush=True)
    deadline = time.time() + hours * 3600
    data = (train, dev, dtrain, dtest, ditems, dpairs, val)
    last = None
    while True:
        start = time.time()
        day(brain, data, rng, pages, teacher_n, self_n, talk_n)
        brain.save(state)
        took = time.time() - start
        last = took
        if time.time() + 1.3 * last > deadline:
            break
    print(f"stopped after day {brain.day}; state saved to {state}", flush=True)
    return brain


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="mode", required=True)
    e = sub.add_parser("experiment")
    e.add_argument("--seeds", type=int, default=5)
    e.add_argument("--workers", type=int, default=4)
    e.add_argument("--dev-limit", type=int, default=None)
    e.add_argument("--teacher", type=int, default=1000)
    e.add_argument("--self", dest="self_n", type=int, default=20000)
    e.add_argument("--talk", type=int, default=20000)
    e.add_argument("--out", default="results/selftrain")
    lp = sub.add_parser("loop")
    lp.add_argument("--hours", type=float, default=9.0)
    lp.add_argument("--state", default="state")
    lp.add_argument("--resume", default=None)
    lp.add_argument("--pages", type=int, default=12)
    lp.add_argument("--teacher", type=int, default=3000)
    lp.add_argument("--self", dest="self_n", type=int, default=4000)
    lp.add_argument("--talk", type=int, default=3000)
    a = ap.parse_args()
    if a.mode == "experiment":
        baselines, results = experiment(a.out, a.seeds, a.workers, a.dev_limit, a.teacher, a.self_n, a.talk)
        verdict = analyse(baselines, results)
        json.dump(verdict, open(os.path.join(a.out, "preregistered.json"), "w"), indent=1)
        print(json.dumps(verdict, indent=1))
    else:
        loop(a.state, a.hours, a.resume, a.pages, a.teacher, a.self_n, a.talk)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/brain/chat/web.py
"""Looking things up.

When the brain is asked about something it has never been told, it can go and read about it. Wikipedia is used
rather than a general search engine: the prose is clean, the licence is clear, and each fact keeps a source, so
anything the brain learned this way can be traced back and removed.

What comes back is *text*, and it is treated as text - sentences to read with the same reader used on everything
else, never as instructions. The brain has no way to act on what a page says even if a page tried to tell it to;
the only thing a page can do is add relations to a map, and every one of them is attributed and reversible.
"""
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request

API = "https://en.wikipedia.org/w/api.php"
# Wikimedia asks automated clients to say who they are and how to reach whoever runs them, and refuses or
# throttles generic agents from shared cloud addresses. v17 read 0 encyclopedia sentences on Kaggle with the
# old agent string, which had no contact in it.
UA = ("NeuroMoE-research/0.2 (https://www.kaggle.com/ericdu4831; gradient-free language research) "
      "python-urllib")
PACE = 0.25         # seconds between requests: a batch of hundreds should not arrive as one burst
LAST_ERROR = {}     # the most recent failure, so a caller that got nothing can say why
_last = [0.0]
SENT = re.compile(r"(?<=[.!?])\s+")


def _get(params, timeout=20, tries=4):
    """One request, waiting longer each time Wikipedia asks to be left alone. A 429 is a request to back off,
    not an error to retry immediately, and a shared service deserves that courtesy."""
    url = API + "?" + urllib.parse.urlencode({**params, "format": "json"})
    delay = 1.5
    for attempt in range(tries):
        wait = PACE - (time.time() - _last[0])
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.time()
        req = urllib.request.Request(url, headers={"User-Agent": UA})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            body = ""
            try:
                body = e.read(300).decode("utf-8", "ignore")
            except Exception:
                pass
            LAST_ERROR.update(code=e.code, body=" ".join(body.split())[:200], url=API)
            if e.code in (429, 503) and attempt < tries - 1:
                after = e.headers.get("Retry-After")
                time.sleep(min(float(after) if (after or "").isdigit() else delay, 30.0))
                delay *= 2
                continue
            raise
        except Exception as e:
            LAST_ERROR.update(code=type(e).__name__, body=str(e)[:200], url=API)
            if attempt < tries - 1:
                time.sleep(delay)
                delay *= 2
                continue
            raise
    raise RuntimeError("gave up after retries")


SUGGESTED = {}      # query -> the spelling the search engine thinks was meant, from the last search


def search(query, limit=5):
    """-> [(title, snippet)] - what pages exist about this."""
    try:
        d = _get({"action": "query", "list": "search", "srsearch": query, "srlimit": limit,
                  "srinfo": "suggestion"})
    except Exception as e:
        return [], f"{type(e).__name__}: {e}"
    suggestion = d.get("query", {}).get("searchinfo", {}).get("suggestion")
    if suggestion:
        SUGGESTED[query] = suggestion
    hits = [(h["title"], re.sub("<[^>]+>", "", h.get("snippet", ""))) for h in d.get("query", {}).get("search", [])]
    return hits, None


def read_page(title, sentences=12, intro_only=False):
    """-> (sentences, url, error). A page as plain sentences.

    The intro alone says what a thing *is* and almost nothing about what it did, which is where the links to
    other things live - so by default this reads on into the body. It is still only the first few hundred
    sentences, and the caller decides how many of them to look at.
    """
    params = {"action": "query", "prop": "extracts", "titles": title, "explaintext": 1, "redirects": 1}
    if intro_only:
        params["exintro"] = 1
    try:
        d = _get(params)
    except Exception as e:
        return [], None, f"{type(e).__name__}: {e}"
    pages = d.get("query", {}).get("pages", {})
    for _, p in pages.items():
        text = (p.get("extract") or "").strip()
        if not text:
            continue
        url = "https://en.wikipedia.org/wiki/" + urllib.parse.quote(p["title"].replace(" ", "_"))
        out = [s.strip() for s in SENT.split(text) if 20 < len(s.strip()) < 400]
        return out[:sentences], url, None
    return [], None, "no extract"


def _slips(a, b):
    """Edit distance: how many keystrokes separate what was typed from what the engine suggests."""
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def _words(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


def choose(query, hits, context=()):
    """Which search hit is the page about the thing asked for.

    Taking the first hit is how "that compares and contrasts the three cities" read a book about the history
    of technology, and how "kl" read the Kullback-Leibler divergence. A page is the right one when its title
    *is* what was asked about - most of the query's words in the title, few extra - and, where the query is
    ambiguous, when its summary mentions what the conversation is about. Lists, disambiguation pages and
    "outline of" pages are never the answer to "what is X".
    """
    q = _words(query)
    ctx = _words(" ".join(context))
    # "the capital city of India" is a description, not a name. Its answer is Delhi, whose title shares no
    # word with it, and scoring titles by overlap picked the article on capital cities in general. When the
    # query describes rather than names, the search engine's own ranking is the better judge.
    describes = len(q) > 3 and bool(q & {"of", "in", "the", "which", "who", "largest", "biggest", "first",
                                         "capital", "founder", "author", "inventor", "president", "leader"})
    best, best_score = None, float("-inf")
    for rank, (title, snippet) in enumerate(hits):
        low = title.lower()
        if "disambiguation" in low or low.startswith(("list of", "lists of", "outline of", "index of",
                                                      "glossary of", "timeline of")):
            continue
        t = _words(title)
        covered = len(q & t) / max(1, len(q))
        extra = len(t - q) / max(1, len(t))
        score = 3.0 * covered - 1.0 * extra - 0.3 * rank + 0.5 * len(ctx & _words(snippet))
        if describes:
            score = -rank + 0.5 * len(ctx & _words(snippet))
        if low == query.lower().strip():
            score += 2.0
        if score > best_score:
            best, best_score = title, score
    return best or (hits[0][0] if hits else None)


def look_up(query, sentences=12, context=()):
    """Search, choose the page actually about the query, read it. -> dict with title, url, sentences, error."""
    hits, err = search(query, limit=6)
    if err:
        return {"query": query, "error": err, "sentences": [], "title": None, "url": None}
    # a name this system has never read - "beijig" - is one the search engine has seen misspelled a million
    # times. When none of the hits even contains what was typed and the engine offers a spelling, take it:
    # that is the last line of hearing past a typo, for the words the corpus was too small to know.
    meant = SUGGESTED.pop(query, None)
    # only a respelling, not a different word: "beijig" -> "beijing" is one slip, "auifh" -> "arith" is a
    # guess, and following a guess lands on a page about something nobody asked about
    if meant and _slips(query.lower(), meant.lower()) > max(1, len(query) // 5):
        meant = None
    if meant and meant.lower() != query.lower() and not any(query.lower() in t.lower() for t, _ in hits):
        better, err2 = search(meant, limit=6)
        if better and not err2:
            hits, query = better, meant
    if not hits:
        return {"query": query, "error": "nothing found", "sentences": [], "title": None, "url": None}
    title = choose(query, hits, context)
    sents, url, err = read_page(title, sentences)
    # a word with several meanings lands on a page listing them. The conversation usually says which one -
    # a chat about planets wants Mercury the planet - so the search is run again with that as a hint.
    ambiguous = not sents or re.search(r"\bmay (also )?refer to\b|\bcan refer to\b", " ".join(sents[:2]), re.I)
    if ambiguous:
        for hint in [c for c in context if c][:3] or ["topic"]:
            more, _ = search(f"{query} {hint}", limit=6)
            more = [h for h in (more or []) if h[0] != title]       # not the page of meanings again
            better = choose(f"{query} {hint}", more, context) if more else None
            if better and better != title:
                s2, u2, e2 = read_page(better, sentences)
                if s2 and not re.search(r"\bmay (also )?refer to\b", " ".join(s2[:2]), re.I):
                    title, sents, url, err = better, s2, u2, e2
                    break
    return {"query": query, "title": title, "url": url, "sentences": sents, "error": err,
            "alternatives": [h[0] for h in hits if h[0] != title][:4]}


if __name__ == "__main__":
    import sys
    r = look_up(" ".join(sys.argv[1:]) or "octopus")
    print(r["title"], "->", r["url"], r.get("error") or "")
    for s in r["sentences"][:6]:
        print("  -", s[:110])

In [ ]:
%%writefile /kaggle/working/brain/PREREGISTRATION_selftrain.md
# Pre-registration: can the brain teach itself to read and to talk?

Written 2026-09-23, before the first Kaggle run of `brain/selftrain.py`. Code at the commit this file is in.

## Why

Kaggle v17 and the reading rerun gave near-identical transcripts from a 0-sentence and a 444,621-sentence
corpus: the replies were templates around a retrieved sentence, and nothing learned decided anything. The
redesign makes every reply a *choice* made by a striatum whose weights are learned by practice with a
three-factor (dopamine-gated, delta-rule) update - no backpropagation, no hidden layers - and adds a loop in
which the brain reads, makes its own practice questions, sleeps (consolidates, downscales, prunes) and sits
held-out exams.

## Exploratory pilots (local, one seed, reported so nothing is hidden)

- Reading, SQuAD dev (2,000-question sample): 1,000 teacher questions F1 0.368; 20,000 self-made questions
  and no teacher F1 0.305; both F1 0.388. 20,000 teacher questions F1 0.464 (1,000-question sample).
- Talking, DailyDialog test (2,000 items, 1 of 10): overlap 0.310, association 0.361, recall 0.527, learned
  weighting after 10,000 practice exchanges 0.628. An untrained striatum 0.102 (chance 0.1).
- A bug found in piloting: with the real reply always first, ties went to it and an untrained brain scored
  1.000. Fixed before this file was written (fixed shuffled positions, random tie-breaking).

## Design

Reading exam: all 10,570 SQuAD v1.1 dev questions, passage given, official EM/F1 normalisation. Never practised
on, never tuned on. Talking exam: all 6,740 DailyDialog test exchanges, the real reply among nine others drawn
from the test set, fixed seed 7, positions shuffled; R@1.

Arms, five seeds each (seed = which teacher questions, practice order, cloze randomness):

| arm | teacher questions | self-made questions | paraphrase by meaning codes |
|---|---|---|---|
| teacher | 1,000 | 0 | - |
| teacher+self | 1,000 | 20,000 | yes (20% of content words) |
| self | 0 | 20,000 | yes |
| teacher+self-noswap | 1,000 | 20,000 | no |
| talk | 20,000 practice exchanges from DailyDialog train | | |

Self-made questions come from SQuAD *train* passages whose questions were not given to that seed as teacher
questions (their questions are never seen). Word rarity and meaning codes are read from SQuAD train passages.

Baselines, computed once (deterministic): sliding window (Richardson et al. 2013, as run in the SQuAD paper,
reimplemented over the same candidates); for talking, each single signal alone (overlap/TF-IDF, cross-turn
association, episodic recall). Literature anchors, not re-run: SQuAD sliding window ~20 F1, logistic
regression 51.0 F1 (Rajpurkar et al. 2016).

## Hypotheses

- **H1 (primary).** teacher+self > teacher on dev F1. Does practising on its own questions add to scarce
  teaching?
- **H2.** self > sliding window on dev F1. Can it learn to read with no teacher at all? The comparison value is
  the *higher* of our reimplementation (11.6 F1 on a 300-question smoke test, weaker than published) and the
  published dev figure of 20.2 F1.
- **H3.** teacher+self > teacher+self-noswap on dev F1. Does paraphrasing its own questions with learned word
  meanings help? (The pilot difference was 0.6 points; this may well fail.)
- **H4.** talk > the best single signal on R@1. Does the learned striatum beat the best thing it is built from?

## Analysis

H1, H3: paired t across the five seeds (df 4). H2, H4: one-sample t of the five seeds against the fixed
baseline value. Two-sided, alpha 0.05, Holm-Bonferroni across H1-H4. Effect sizes reported as differences in
F1 / R@1 with the per-seed values.

## The self-training loop (descriptive, not a hypothesis test)

After the experiment the same notebook runs `selftrain.py loop` from an untrained brain until the session
budget: each "day" reads ~12 Wikipedia pages chosen by curiosity, practises 3,000 teacher questions, ~4,000
self-made questions by prioritised replay, and 3,000 exchanges, sleeps (downscale 1%, prune below 1e-4) and
sits both exams. The learning curve is reported as is. Teacher questions run out at day ~29; the days after
that measure whether self-practice alone keeps the exam moving.

## What would count as failure

If H1 fails, self-practice does not add to teaching at this budget and the loop's self-practice is decoration;
if H2 fails, it cannot learn to read without a teacher. Either is reported as such.

In [ ]:
%%writefile /kaggle/working/brain/tests/check_selftrain.py
"""The redesigned parts, checked cheaply: the striatum learns, the reader reads, the brain asks itself questions,
exams cannot be passed by accident, and a brain saved is the brain loaded."""
import os
import random
import sys
import tempfile

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path[:0] = [ROOT, os.path.join(ROOT, "chat")]

from brain import cloze  # noqa: E402
from brain.reading import Rarity, Reader, f1, exact, normalize  # noqa: E402
from brain.striatum import Striatum  # noqa: E402
from brain.selftrain import Brain, dialogue_exam_items, pick  # noqa: E402

FAILED = []


def check(name, cond, detail=""):
    print(f"{'PASS' if cond else 'FAIL'}  {name}" + (f"   {detail}" if detail and not cond else ""))
    if not cond:
        FAILED.append(name)


# striatum: a choice rewarded consistently is learned, and the untrained striatum has no preference
st = Striatum(bits=12, lr=0.2)
enc = st.encode([["a", "shared"], ["b", "shared"], ["c", "shared"]])
check("an untrained striatum scores every option the same", len(set(st.scores(enc).round(9))) == 1)
for _ in range(30):
    st.teach(enc, [0.0, 1.0, 0.0])
check("taught option wins after practice", st.choose(enc)[0] == 1)
before = st.synapses()
st.sleep(downscale=0.5, floor=0.5)
check("sleep prunes weak synapses", st.synapses() <= before)

# scoring follows the official SQuAD normalisation
check("EM ignores case, articles and punctuation", exact("The Eiffel Tower.", "eiffel tower") == 1.0)
check("F1 gives partial credit", 0 < f1("Saint Bernadette", "Saint Bernadette Soubirous") < 1)
check("normalize drops articles", normalize("a cat and the dog") == "cat and dog")

# the reader: after practice on one question shape, it answers another of the same shape
passage = ("The Amazon River flows through Brazil. It was first explored by Francisco de Orellana in 1542. "
           "The river is about 6,400 kilometres long.")
rar = Rarity()
rar.read(passage)
reader = Reader(rar)
qa = Striatum(bits=16, lr=0.2)
for _ in range(20):
    reader.practise(qa, "When was the river first explored?", passage, ["1542"])
    reader.practise(qa, "Who first explored the river?", passage, ["Francisco de Orellana"])
ans, _, _ = reader.answer(qa, "When did anyone first explore the Amazon?", passage)
check("reader answers a paraphrased when-question", "1542" in ans, ans)

# it asks itself questions whose answers are in the text
qs = cloze.make(passage, random.Random(0))
check("self-made questions exist", len(qs) >= 2, str(qs))
check("and every answer is a stretch of the passage", all(a in passage for _, a in qs), str(qs))
check("and every question starts with a wh-word", all(q.split()[0].lower() in ("what", "who", "when", "where", "how")
                                                     for q, _ in qs), str(qs))

# the dialogue exam cannot be passed by a flat score
items = dialogue_exam_items([[f"line {i}", f"reply {i}", f"more {i}"] for i in range(12)])
golds = {g for _, _, g in items}
check("the real reply is not always in the same place", len(golds) > 1, str(golds))
rng = random.Random(0)
flat = sum(pick(np.zeros(10), rng) == g for _, _, g in items * 50) / (len(items) * 50)
check("a flat score scores near chance", flat < 0.25, f"{flat:.2f}")

# a saved brain is the brain loaded
b = Brain()
b.store(passage, "Amazon")
b.qa = qa
d = tempfile.mkdtemp()
b.save(d)
c = Brain.load(d, [["hi", "hello there"], ["how are you", "fine"], ["hi", "hello"]])
check("reading weights survive a save", np.allclose(b.qa.consolidated(), c.qa.consolidated()))
check("memories survive a save", c.passages[0]["text"] == passage)

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

### Does it work at all

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "tests/check_selftrain.py"], capture_output=True, text=True, cwd="/kaggle/working/brain")
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "checks failed - not training a broken brain"

### Self-training, for the rest of the session

In [ ]:
import glob, subprocess, sys
# the self-training loop for the rest of the session, resuming from an earlier session's checkpoint if given one
previous = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/brain.pkl", recursive=True)]
resume = ["--resume", previous[0]] if previous else []
print("resuming from", previous[0] if previous else "nothing - a newborn brain")
left = 11.2 - (time.time() - START) / 3600
print(f"{left:.2f} hours left for the loop")
p = subprocess.Popen([sys.executable, "-u", "brain/selftrain.py", "loop", "--hours", f"{max(0.2, left):.2f}",
                      "--state", "/kaggle/working/state", *resume],
                     cwd="/kaggle/working/brain", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()

### The learning curve

In [ ]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
log = json.load(open("/kaggle/working/state/log.json"))
days = [e["day"] for e in log]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(days, [e["exam_f1"] for e in log], "o-", label="F1")
ax[0].plot(days, [e["exam_em"] for e in log], "s-", label="EM")
ax[0].axhline(0.202, ls=":", c="gray"); ax[0].text(0, 0.21, "sliding window (published)", fontsize=8)
ax[0].axhline(0.51, ls="--", c="gray"); ax[0].text(0, 0.52, "logistic regression (published)", fontsize=8)
ax[0].set_title("reading: SQuAD dev"); ax[0].set_xlabel("day"); ax[0].legend()
ax[1].plot(days, [e["exam_r_at_1"] for e in log], "o-", c="C2")
ax[1].axhline(0.1, ls=":", c="gray"); ax[1].set_title("talking: DailyDialog test, R@1 of 10"); ax[1].set_xlabel("day")
plt.tight_layout(); plt.savefig("/kaggle/working/learning_curve.png", dpi=120)
for e in log:
    print(e["day"], e.get("exam_f1"), e.get("exam_r_at_1"), e.get("teacher_seen"), e.get("passages"),
          (e.get("pages") or [])[:4])